In [12]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "agents").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from agents.ontology_qa_agent import build_ontology_qa_agent

In [13]:
DATA_PATH = PROJECT_ROOT / "data" / "test_questions_v1_0_manual.json"
samples = json.loads(DATA_PATH.read_text(encoding="utf-8"))

sample_index = 17
sample = samples[sample_index]

print(sample["question"])
print(json.dumps(sample["options"], ensure_ascii=False, indent=2))

Số lượng trường đại học ở Việt Nam
[
  {
    "id": 1,
    "text": "66"
  },
  {
    "id": 2,
    "text": "67"
  },
  {
    "id": 3,
    "text": "68"
  },
  {
    "id": 4,
    "text": "69"
  },
  {
    "id": 5,
    "text": "70"
  }
]


In [14]:
graph = build_ontology_qa_agent(max_iterations=15)

In [15]:
graph_input = {
    "question": sample["question"],
    "options": sample["options"],
}

final_output = None
for event in graph.stream(graph_input, stream_mode="updates"):
    for node_name, update in event.items():
        print(f"\n{'=' * 20} {node_name} {'=' * 20}")

        if node_name == "agent":
            message = update["messages"][-1]
            print("content:", message.content)
            print("tool_calls:", json.dumps(message.tool_calls, ensure_ascii=False, indent=2, default=str))
        elif node_name == "tools":
            for message in update.get("messages", []):
                print(f"{message.name}: {message.content}")
        elif node_name == "post_tool":
            print("generated_sparqls:")
            for query in update.get("generated_sparqls", []):
                print(repr(query))
            print("executions:", json.dumps(update.get("executions", []), ensure_ascii=False, indent=2, default=str))
        elif node_name == "initialize":
            print("initialized keys:", sorted(update))
        elif node_name == "retry":
            print(update["messages"][-1].content)
        else:
            print(json.dumps(update, ensure_ascii=False, indent=2, default=str))

        if node_name == "finalize":
            final_output = {
                "selected_option_id": update.get("selected_option_id"),
                "answer": update.get("answer"),
                "result": update.get("result"),
            }

print("\nFINAL OUTPUT")
print(json.dumps(final_output, ensure_ascii=False, indent=2, default=str))


==================== initialize ====================
initialized keys: ['actions', 'executions', 'generated_sparqls', 'iteration_count', 'messages', 'steps']

==================== agent ====================
content: 
tool_calls: [
  {
    "name": "search_class_by_label",
    "args": {
      "query": "university"
    },
    "id": "chatcmpl-tool-bc7f0f13af06fd25",
    "type": "tool_call"
  }
]

==================== tools ====================
search_class_by_label: [{"label": "University", "uri": "http://dbpedia.org/ontology/University", "kind": "class", "score": 44.76139, "comments": [], "domain": "", "range": "", "domain_label": "", "range_label": ""}]

==================== post_tool ====================
generated_sparqls:
executions: []

==================== agent ====================
content: 
tool_calls: [
  {
    "name": "search_entity_by_label",
    "args": {
      "query": "Vietnam"
    },
    "id": "chatcmpl-tool-befeeb6c43ae74da",
    "type": "tool_call"
  }
]

================